# EDA

Raw Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

: 

In [ ]:
df = pd.read_csv('aug_train.csv')

Understand Data

In [ ]:
display(df.head())

In [ ]:
print("Dataset Shape:", df.shape)

In [ ]:
print("Columns:")
print(df.columns.tolist())

In [ ]:
print("Dataset Information:")
df.info()

Statistical Analysis

In [ ]:
print("Statistical Description:")
display(df.describe())

Missing Value Check

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

Duplicates Check

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

visualization

In [ ]:
print("Target Distribution:")
print(df['target'].value_counts())

In [ ]:
df['target'].value_counts().plot(kind='bar')

plt.xlabel('Target')
plt.ylabel('Number of Candidates')
plt.title('Target Distribution')
plt.show()

categorical&numberical Analisis

In [ ]:
numeric_features = df.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = df.select_dtypes(
    include=['object']
).columns.tolist()

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Visualization

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df['training_hours'].dropna(), bins=30)

plt.xlabel('Training Hours')
plt.ylabel('Frequency')
plt.title('Distribution of Training Hours')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df['city_development_index'].dropna(), bins=30)

plt.xlabel('City Development Index')
plt.ylabel('Frequency')
plt.title('Distribution of City Development Index')

plt.show()

outliers Detection

In [ ]:
plt.figure(figsize=(8, 5))

plt.boxplot(df['training_hours'].dropna())

plt.ylabel('Training Hours')
plt.title('Training Hours - Boxplot')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.boxplot(df['city_development_index'].dropna())

plt.ylabel('City Development Index')
plt.title('City Development Index - Boxplot')

plt.show()

Correlation/Relationships

In [ ]:
plt.figure(figsize=(8, 5))

df.boxplot(
    column='training_hours',
    by='target'
)

plt.xlabel('Target')
plt.ylabel('Training Hours')
plt.title('Training Hours by Target')

plt.suptitle('')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

df.boxplot(
    column='city_development_index',
    by='target'
)

plt.xlabel('Target')
plt.ylabel('City Development Index')
plt.title('City Development Index by Target')

plt.suptitle('')

plt.show()

identify proplem

اكتشفت من خلال تحليل البيانات ان ال
 target  غير متوازن
مع وجود بعض
outliers .في بعض . features  ال

# Data preprocessing

libraries

           Encoding Original Data

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
df_encoded = df.copy()

categorical_features = df_encoded.select_dtypes(include=['object']).columns

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

encoded_data = encoder.fit_transform(df_encoded[categorical_features])

encoded_columns = encoder.get_feature_names_out(categorical_features)

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoded_columns,
    index=df_encoded.index
)

numeric_df = df_encoded.drop(columns=categorical_features)

df_encoded = pd.concat([numeric_df, encoded_df], axis=1)

display(df_encoded.head())
print("Original shape:", df.shape)
print("Encoded shape:", df_encoded.shape)

       Scale Original Numerical Data

In [ ]:
from sklearn.preprocessing import StandardScaler

df_scaled = df.copy()

numeric_columns = df_scaled.select_dtypes(include=['int64', 'float64']).columns

scaler_df = StandardScaler()

df_scaled[numeric_columns] = scaler_df.fit_transform(
    df_scaled[numeric_columns]
)

display(df_scaled.head())

Compare Before and After Scaling

In [ ]:
print("Before Scaling:")
display(df[numeric_columns].head())

print("After Scaling:")
display(df_scaled[numeric_columns].head())

print("\nMean after scaling:")
print(df_scaled[numeric_columns].mean())

print("\nStandard deviation after scaling:")
print(df_scaled[numeric_columns].std())

        Handle Duplicates

In [ ]:
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

        Handle Outliers

In [ ]:
def handle_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers_before = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ].shape[0]

    df[column] = df[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    outliers_after = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ].shape[0]

    print(f"{column}")
    print(f"Outliers before handling: {outliers_before}")
    print(f"Outliers after handling: {outliers_after}")
    print("-" * 40)

    return df

In [ ]:
df = handle_outliers(df, 'training_hours')
df = handle_outliers(df, 'city_development_index')

        outliers visualization

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)

sns.boxplot(
    data=df,
    y='training_hours',
    color='skyblue'
)

plt.title('Training Hours After Outlier Handling')

plt.subplot(1, 2, 2)

sns.boxplot(
    data=df,
    y='city_development_index',
    color='lightgreen'
)

plt.title('City Development Index After Outlier Handling')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

sns.stripplot(
    data=df,
    y='training_hours',
    color='purple',
    jitter=0.2,
    size=5
)

plt.title('Training Hours After Outlier Handling')

plt.subplot(1, 2, 2)

sns.stripplot(
    data=df,
    y='city_development_index',
    color='purple',
    jitter=0.2,
    size=5
)

plt.title('City Development Index After Outlier Handling')

plt.tight_layout()
plt.show()

         Feature Engineering / Selection

                    Separate Features and Target

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

                  Remove ID Column

In [ ]:
X = X.drop('enrollee_id', axis=1)

print("Features after removing ID:")
print(X.columns.tolist())

               Feature Engineering

In [ ]:
X['experience_years'] = X['experience'].replace({
    '<1': 0.5,
    '>20': 21
}).astype(float)

X['high_experience'] = (X['experience_years'] >= 5).astype(int)

X['training_per_experience'] = X['training_hours'] / (X['experience_years'] + 1)

X['large_company'] = X['company_size'].isin([
    '1000-4999',
    '5000-9999',
    '10000+'
]).astype(int)

print("Feature Engineering completed successfully!")

print(X[['experience_years', 'high_experience',
         'training_per_experience', 'large_company']].head())

      Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

     Identify Numerical & Categorical Features

In [ ]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

         Handle Missing Values

        Transformation-encode-scaling

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

balance

In [ ]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_balanced, y_train_balanced = ros.fit_resample(X_train, y_train)

print("Before Balancing:")
print(y_train.value_counts())

print("\nAfter Balancing:")
print(y_train_balanced.value_counts())

     Apply Full Preprocessing

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train_balanced)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train Shape:", X_train_processed.shape)
print("Processed X_test Shape:", X_test_processed.shape)

    Convert Processed Data to DataFrames

In [ ]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed= pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train_balanced.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)
print("===== Data After Encoding =====")

print(X_train_processed.head())

       Check Missing Values After Processing

In [ ]:
print("Missing values in X_train:",
      X_train_processed.isnull().sum().sum())

print("Missing values in X_test:",
      X_test_processed.isnull().sum().sum())

      final data

In [ ]:
print("===== FINAL DATA =====")

print("X_train:", X_train_processed.shape)
print("X_test:", X_test_processed.shape)
print("y_train:", y_train_balanced.shape)
print("y_test:", y_test.shape)

print("\nData preprocessing completed successfully!")

        Save Prepared Data

In [ ]:
X_train_processed.to_csv('X_train_processed.csv', index=False)
X_test_processed.to_csv('X_test_processed.csv', index=False)
y_train_balanced.to_csv('y_train_balanced.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Prepared datasets saved successfully!")